In [1]:
import pandas as pd
from pathlib import Path

# On essaie d'utiliser __file__, sinon on prend le dossier actuel
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

ROOT = BASE_DIR.parent
DATA_DIR = ROOT / "data"

%ls ../data/extracted/
print('-'*30)
%ls ../data/processed/data/commune

communes_31_geom_simplified.parquet
communes_31_geom_simplified_geo.parquet
communes_densite.csv
dept_taux_mortalité_2024.parquet
mortalite_departements.csv
raw_comm_acces_urgences.csv
raw_comm_acces_urgences.parquet
raw_dept_taux_mortalité_2024.csv
raw_dept_taux_mortalité_2024.parquet
------------------------------
commune_clusters.parquet               raw_comm_score_apl_niv1.parquet
commune_niv1_score_irdes.csv           raw_comm_score_apl_niv2.parquet
commune_niv1_score_irdes.parquet       raw_comm_score_apl_niv3.parquet
raw_comm_score_apl_niv0.csv            score_sante_territoires_final.csv
raw_comm_score_apl_niv0.parquet        score_sante_territoires_final.parquet


In [ ]:
file_path = DATA_DIR / 'processed/data/commune/raw_comm_score_apl_niv3.parquet'

df = pd.read_parquet(file_path)

df.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,score_apl,quintile_apl_nat,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region,tps_SU_SMUR,tx_mortalite,tx_mort_premature
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832.0,35.270,122.416,...,-0.317946,Q2 (faible),859.0,1590.0,01,200069193,84,31.0,7.9,1.4
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267.0,36.427,109.849,...,-0.291066,Q2 (faible),273.0,920.0,01,240100883,84,18.0,7.9,1.4
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854.0,59.621,201.121,...,0.778222,Q5 (très bon),15554.0,2460.0,01,240100883,84,0.0,7.9,1.4
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897.0,50.539,131.876,...,0.583430,Q5 (très bon),1917.0,1590.0,01,200042497,84,23.5,7.9,1.4
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113.0,10.929,44.227,...,-1.231006,Q1 (très faible),114.0,590.0,01,200040350,84,15.0,7.9,1.4


In [3]:
# Standardiser 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

for col in ['tps_SU_SMUR', 'tx_mortalite', 'tx_mort_premature']:
    df[col + '_std'] = scaler.fit_transform(df[[col]])

In [6]:
df.columns

Index(['code_insee_comm', 'nom_commune', 'apl_medecins', 'apl_65ans',
       'apl_62ans', 'apl_60ans', 'pop_standard_2021', 'pop_totale_2021',
       'apl_dentistes', 'apl_infirmiers', 'apl_kines', 'apl_sagesfemmes',
       'apl_medecins_2022', 'apl_dentistes_2022', 'apl_infirmiers_2022',
       'apl_kines_2022', 'apl_sagesfemmes_2022', 'delta_apl_medecins',
       'delta_apl_dentistes', 'delta_apl_infirmiers', 'delta_apl_kines',
       'delta_apl_sagesfemmes', 'apl_medecins_std', 'apl_dentistes_std',
       'apl_infirmiers_std', 'apl_kines_std', 'apl_sagesfemmes_std',
       'score_apl', 'quintile_apl_nat', 'population', 'superficie',
       'code_insee_du_departement', 'codes_siren_des_epci',
       'code_insee_de_la_region', 'tps_SU_SMUR', 'tx_mortalite',
       'tx_mort_premature', 'tps_SU_SMUR_std', 'tx_mortalite_std',
       'tx_mort_premature_std'],
      dtype='object')

In [8]:
df_final = df[['code_insee_comm','nom_commune', 'population', 'superficie',
               'apl_medecins', 'apl_65ans', 'apl_dentistes', 'apl_infirmiers', 'apl_kines', 'apl_sagesfemmes',
               'delta_apl_medecins', 'delta_apl_dentistes', 'delta_apl_infirmiers', 'delta_apl_kines', 'delta_apl_sagesfemmes',
               'apl_medecins_std', 'apl_dentistes_std', 'apl_infirmiers_std', 'apl_kines_std', 'apl_sagesfemmes_std',
               'score_apl', 'quintile_apl_nat', 
               'code_insee_du_departement', 'codes_siren_des_epci','code_insee_de_la_region',
               'tps_SU_SMUR', 'tx_mortalite','tx_mort_premature',
               'tps_SU_SMUR_std', 'tx_mortalite_std', 'tx_mort_premature_std'
               ]].copy()

In [10]:
# On prend l'opposé pour que le manque de médecins augmente le score
df_final['apl_fragilite_std'] = df_final['score_apl'] * -1

df_final['score_fragilite_global'] = (
    df_final['apl_fragilite_std'] + 
    df_final['tps_SU_SMUR_std'] + 
    df_final['tx_mort_premature_std']
) / 3

In [11]:
df_final.columns

Index(['code_insee_comm', 'nom_commune', 'population', 'superficie',
       'apl_medecins', 'apl_65ans', 'apl_dentistes', 'apl_infirmiers',
       'apl_kines', 'apl_sagesfemmes', 'delta_apl_medecins',
       'delta_apl_dentistes', 'delta_apl_infirmiers', 'delta_apl_kines',
       'delta_apl_sagesfemmes', 'apl_medecins_std', 'apl_dentistes_std',
       'apl_infirmiers_std', 'apl_kines_std', 'apl_sagesfemmes_std',
       'score_apl', 'quintile_apl_nat', 'code_insee_du_departement',
       'codes_siren_des_epci', 'code_insee_de_la_region', 'tps_SU_SMUR',
       'tx_mortalite', 'tx_mort_premature', 'tps_SU_SMUR_std',
       'tx_mortalite_std', 'tx_mort_premature_std', 'apl_fragilite_std',
       'score_fragilite_global'],
      dtype='object')

In [13]:
file_path = '../data/processed/data/commune/final_comm_indic_projet.parquet'
df_final.to_parquet(file_path)

file_path = '../data/processed/data/commune/final_comm_indic_projet.csv'
df_final.to_csv(file_path)